# EDA 02 · Analiza Zbioru HF Lung

<div style="border-left: 4px solid #2b6cb0; padding: 12px 16px; margin-bottom: 20px; border-radius: 0 4px 4px 0;">
  <strong>Praca Inżynierska:</strong> Analiza i klasyfikacja dźwięków z wykorzystaniem uczenia maszynowego <br>
  <strong>Autorzy:</strong> Norbert Gwiazda, Mateusz Ciurzyński | <strong>Data:</strong> 2026-09-02<br>
  <strong>Zbiór danych:</strong> Respiratory Sound Database (ICBHI 2017)<br>
  <strong>Cel:</strong> Analiza eksploracyjna zbiorów danych
</div>

---

## Spis treści
- [1. Kontekst i Charakterystyka Bazy Danych](#1-kontekst-i-charakterystyka-bazy-danych)
- [2. Środowisko i Import Bibliotek](#2-środowisko-i-import-bibliotek)
- [3. Wczytanie danych](#3-wczytanie-danych)

---

## 1. Kontekst i Charakterystyka Bazy Danych

Zbiór *HF_Lung_V2* (stanowiący rozwinięcie zbioru *HF_Lung_V1*) zawiera cyfrowe nagrania odsłuchowe płuc zebrane przez tajwańskie zespoły naukowe i medyczne. Głównymi podmiotami odpowiedzialnymi za stworzenie i adnotację zbioru są
- **Zespół badawczy i przemysłowy:** *Heroic Faith Medical Science Co., Ltd.* (Tajpej) we współpracy z *Graduate Institute of Biomedical Electronics and Bioinformatics* na Narodowym Uniwersytecie Tajwańskim (NTU).
- **Zespoły kliniczne:** *Far Eastern Memorial Hospital* (**FEMH**) w New Taipei – nagrania w Szpitalu FEMH oraz na oddziałach opieki oddechowej (**RCC/RCW**) – a także baza danych pozyskana z *Taiwan Society of Emergency and Critical Care Medicine* (**TSECCM**) [1].

Baza danych zawiera łącznie 3489,25 minuty nagrań (ponad 58 godzin audio) ułożonych w 13 957 pojedynczych $15$-sekundowych plików9. Dane zebrano od 300 pacjentów. Pliki zawierają adnotacje tekstowe ze znacznikami czasowymi określającymi występowanie i czas trwania wdechów, wydechów, ciągłych dodatkowych dźwięków oddechowych (świsty, furczenia) oraz nieciągłych dodatkowych dźwięków oddechowych (trzeszczenia).

**Lokalizacje osłuchu (8 punktów na klatce piersiowej):**

- **L1:** Druga przestrzeń międzyżebrowa w prawej linii środkowo-obojczykowej (2nd ICS, Right MCL)
- **L2:** Piąta przestrzeń międzyżebrowa w prawej linii środkowo-obojczykowej (5th ICS, Right MCL)
- **L3:** Czwarta przestrzeń międzyżebrowa w prawej linii pachowej średniej (4th ICS, Right MAL)
- **L4:** Dziesiąta przestrzeń międzyżebrowa w prawej linii pachowej średniej (10th ICS, Right MAL)
- **L5:** Druga przestrzeń międzyżebrowa w lewej linii środkowo-obojczykowej (2nd ICS, Left MCL)
- **L6:** Piąta przestrzeń międzyżebrowa w lewej linii środkowo-obojczykowej (5th ICS, Left MCL)
- **L7:** Czwarta przestrzeń międzyżebrowa w lewej linii pachowej średniej (4th ICS, Left MAL)
- **L8:** Dziesiąta przestrzeń międzyżebrowa w lewej linii pachowej średniej (10th ICS, Left MAL)

**Sprzęt użyty do badania:**

- **3M Littmann 3200 Electronic Stethoscope** (Littmann 3200) – cyfrowy stetoskop elektroniczny
- **HF-Type-1** – spersonalizowane wielokanałowe urządzenie osłuchowe z dołączanymi czujnikami akustycznymi umożliwiające jednoczesne ciągłe nagrywanie z wielu punktów klatki piersiowej

**Diagnozy i schorzenia obecne u pacjentów w zbiorze:**

- Zapalenie płuc (Pneumonia)
- Przewlekła Obturacyjna Choroba Płuc / Ostre zaostrzenie POChP (COPD / Acute exacerbation of COPD)
- Przewlekła niewydolność oddechowa (Chronic respiratory failure)
- Ostra niewydolność oddechowa (Acute respiratory failure)
- Zapalenie oskrzeli (Bronchitis)
- Astma (Asthma)
- Zespół ostrej niewydolności oddechowej (ARDS – Acute respiratory distress syndrome)
- Rozedma płuc (Emphysema)
- Płyn w opłucnej (Pleural effusion)
- Pylica płuc (Pneumoconiosis)
- Zatorowość płucna (Pulmonary embolism)

---
## 2. Środowisko i Import Bibliotek


In [38]:
%matplotlib inline
%config InlineBackend.figure_format = 'svg'
%config InlineBackend.close_figures = True


import os
import warnings
from datetime import datetime
from itertools import chain
from pathlib import Path

import librosa
import matplotlib.gridspec as gridspec

# Wizualizacja danych
import matplotlib.pyplot as plt

# Przetwarzanie i analiza numeryczna
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import scipy.stats as stats
import seaborn as sns
import statsmodels.api as sm
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from pyfonts import load_google_font
from pypalettes import load_cmap, load_palette
from scipy.signal import find_peaks

# Wyciszenie zbędnych ostrzeżeń
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

# Wczytanie czionek i palet kolorów
dark_two = load_palette("Dark2")
regular = load_google_font("Roboto", weight=400)
medium = load_google_font("Roboto", weight=500)
bold = load_google_font("Roboto", weight=700)
heavy = load_google_font("Roboto", weight=800)

# Ustawienie motywu
sns.set_theme(style="whitegrid")

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42


## 3. Wczytanie danych

W pierwszej kolejności przyjrzymy się zawartości folderu zbioru danych

In [47]:
CACHE_DIR = '.cache'
CACHE_FILE = os.path.join(CACHE_DIR, 'hf_lung_features.parquet')

os.makedirs(CACHE_DIR, exist_ok=True)

if os.path.exists(CACHE_FILE):
    print(f"Wczytywanie pamięci podręcznej z {CACHE_FILE}")
    table = pq.read_table(CACHE_FILE)
    df = table.to_pandas()

BASE_DIR = Path('../data')
DATA_PATH_1 = BASE_DIR / "HF_Lung_V1-master"
DATA_PATH_2 = BASE_DIR / "HF_Lung_V1_IP-main"

TRAIN_PATH_1 = DATA_PATH_1 / "train"
TEST_PATH_1 = DATA_PATH_1 / "test"

TRAIN_PATH_2 = DATA_PATH_2 / "train"
TEST_PATH_2 = DATA_PATH_2 / "test"

audio_files = list(DATA_PATH_1.rglob("*.wav")) + list(DATA_PATH_2.rglob("*.wav"))

print(f"Wszystkich nagrań: {len(audio_files)}")
print("Pierwsze 5 nagrań:")
for f in audio_files[:5]:
    print(f"  - {f.relative_to(BASE_DIR)}")



Wszystkich nagrań: 13957
Pierwsze 5 nagrań:
  - HF_Lung_V1-master/test/steth_20190821_10_31_21.wav
  - HF_Lung_V1-master/test/steth_20190811_10_02_48.wav
  - HF_Lung_V1-master/test/steth_20190811_10_14_30.wav
  - HF_Lung_V1-master/test/steth_20190811_11_09_58.wav
  - HF_Lung_V1-master/test/steth_20190815_13_54_59.wav


Dane podzielone są na zbiór treningowy i testowy, ale na potrzeby analizy połączymy te zbiory w jeden.

In [48]:
if not os.path.exists(CACHE_FILE):
    annotations = []
    file_metadata = []

    all_paths = chain(
        TRAIN_PATH_1.iterdir(),
        TEST_PATH_1.iterdir(),
        TRAIN_PATH_2.iterdir(),
        TEST_PATH_2.iterdir()
    )

    for file in all_paths:
        relative_filename = str(file.relative_to(BASE_DIR))
        rel_path_no_ext = str(file.relative_to(BASE_DIR).with_suffix(""))
        parts = file.stem.split("_")
        if file.suffix == ".wav":
            if len(parts) == 5:
                audio_type, date_str, h, m, s = parts
                date_obj = datetime(
                    int(date_str[:4]),
                    int(date_str[4:6]),
                    int(date_str[6:8]),
                    int(h),
                    int(m),
                    int(s),
                )
                file_metadata.append({
                    "recording_equipment": audio_type,
                    "date": date_obj,
                    "chest_location": None,
                    "truncation": 1,
                    "filename": rel_path_no_ext,
                })
            elif len(parts) == 3:
                audio_type, date_loc_str, truncation = parts
                date_loc_parts = date_loc_str.split("-")
                args = [int(x) for x in date_loc_parts[:-1]]
                date_obj = datetime(
                    year=args[0],
                    month=args[1],
                    day=args[2],
                    hour=args[3],
                    minute=args[4],
                    second=args[5],
                )
                file_metadata.append({
                    "recording_equipment": audio_type,
                    "date": date_obj,
                    "chest_location": date_loc_parts[-1],
                    "truncation": truncation,
                    "filename": rel_path_no_ext,
                })

        elif file.suffix == ".txt":
            if len(parts) not in (4, 6):
                continue
            try:
                df = pd.read_csv(file, sep=r"\s+", names=["type", "start", "end"])
                df["start"] = pd.to_timedelta(df["start"]).dt.total_seconds()
                df["end"] = pd.to_timedelta(df["end"]).dt.total_seconds()

                phases = df[df["type"].isin(["I", "E"])].sort_values("start")
                anomalies = df[~df["type"].isin(["I", "E"])]

                cycles = []
                current_start, current_end = None, None

                for _, row in phases.iterrows():
                    if row["type"] == "I":
                        if current_start is not None:
                            cycles.append({"start": current_start, "end": current_end})
                        current_start = row["start"]
                        current_end = row["end"]
                    elif row["type"] == "E":
                        if current_start is None:
                            current_start = row["start"]
                        current_end = (
                            max(current_end, row["end"]) if current_end else row["end"]
                        )
                if current_start is not None:
                    cycles.append({"start": current_start, "end": current_end})

                cycles_df = pd.DataFrame(cycles)
                if cycles_df.empty:
                    continue

                cycles_df["filename"] = rel_path_no_ext[:-6]
                # Inicjalizacja kolumn anomalii
                for col in ["crackles", "wheezes", "rhonchi", "stridor"]:
                    cycles_df[col] = 0

                for _, anomaly in anomalies.iterrows():
                    a_type = str(anomaly["type"]).lower()
                    # Mapowanie literówek z oryginalnego zbioru na standardowe nazwy
                    type_map = {
                        "d": "crackles",
                        "wheeze": "wheezes",
                        "rhonchi": "rhonchi",
                        "stridor": "stridor",
                    }
                    target_col = type_map.get(a_type)

                    if target_col:
                        mask = (cycles_df["start"] < anomaly["end"]) & (
                            cycles_df["end"] > anomaly["start"]
                        )
                        cycles_df.loc[mask, target_col] = 1

                annotations.append(cycles_df)
            except Exception as e:
                print(f"Pominięto uszkodzony plik adnotacji {file.name}: {e}")


    df_all = pd.concat(annotations, ignore_index=True)
    metadata_df = pd.DataFrame(file_metadata)
    df = pd.merge(df_all, metadata_df, on="filename", how="left")

df.head()

,start,end,filename,crackles,wheezes,rhonchi,stridor,recording_equipment,date,chest_location,truncation
0,4.281,6.572,HF_Lung_V1-master/train/steth_20190622_13_52_19,1,0,0,0,steth,2019-06-22 13:52:19,NaN,1
1,0.342,1.216,HF_Lung_V1-master/train/steth_20181210_08_33_26,0,0,0,0,steth,2018-12-10 08:33:26,NaN,1
2,3.378,4.252,HF_Lung_V1-master/train/steth_20181210_08_33_26,0,0,0,0,steth,2018-12-10 08:33:26,NaN,1
3,6.414,7.287,HF_Lung_V1-master/train/steth_20181210_08_33_26,0,0,0,0,steth,2018-12-10 08:33:26,NaN,1
4,9.468,10.342,HF_Lung_V1-master/train/steth_20181210_08_33_26,0,0,0,0,steth,2018-12-10 08:33:26,NaN,1


W ramach wstępnej analizy załadowane zostanie dodatkowo kilka cech wyciągniętych z plików audio.

In [ ]:
if not os.path.exists(CACHE_FILE):
    rms_dict = {}
    zcr_dict = {}
    spectral_centroid_dict = {}
    spectral_flatness_dict = {}
    spectral_rolloff_dict = {}
    spectral_bandwidth_dict = {}
    crest_factor = {}

    for filename in df['filename'].unique():
        y, sr = librosa.load(BASE_DIR / f"{filename}.wav" , sr=None)
        rms = np.mean(librosa.feature.rms(y=y))
        zcr_dict[filename] = np.mean(librosa.feature.zero_crossing_rate(y=y))
        rms_dict[filename] = rms

        crest_factor[filename]  = np.max(np.abs(y)) / (rms+ 1e-8)

        spectral_centroid_dict[filename]  = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        spectral_bandwidth_dict[filename]  = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
        spectral_rolloff_dict[filename]  = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.85))
        spectral_flatness_dict[filename]  = np.mean(librosa.feature.spectral_flatness(y=y))


    df['audio_rms'] = df['filename'].map(rms_dict)
    df['audio_zcr'] = df['filename'].map(zcr_dict)
    df['crest_factor'] = df['filename'].map(crest_factor)

    df['spectral_centroid'] = df['filename'].map(spectral_centroid_dict)
    df['spectral_bandwidth'] = df['filename'].map(spectral_bandwidth_dict)
    df['spectral_rolloff'] = df['filename'].map(spectral_rolloff_dict)
    df['spectral_flatness'] = df['filename'].map(spectral_flatness_dict)

    df['truncation'] = pd.to_numeric(df['truncation'], errors='coerce').astype('Int64')

    table = pa.Table.from_pandas(df)
    pq.write_table(table, CACHE_FILE)
    print(f"Zapisano dane do buforu {CACHE_FILE}")

ArrowInvalid: ("Could not convert '3' with type str: tried to convert to int64", 'Conversion failed for column truncation with type object')

In [42]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34784 entries, 0 to 34783
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   start               34784 non-null  float64
 1   end                 34784 non-null  float64
 2   filename            34784 non-null  str    
 3   crackles            34784 non-null  int64  
 4   wheezes             34784 non-null  int64  
 5   rhonchi             34784 non-null  int64  
 6   stridor             34784 non-null  int64  
 7   audio_rms           34784 non-null  float32
 8   audio_zcr           34784 non-null  float64
 9   crest_factor        34784 non-null  float32
 10  spectral_centroid   34784 non-null  float64
 11  spectral_bandwidth  34784 non-null  float64
 12  spectral_rolloff    34784 non-null  float64
 13  spectral_flatness   34784 non-null  float32
dtypes: float32(3), float64(6), int64(4), str(1)
memory usage: 4.4 MB


## Literatura
- [1] F.-S. Hsu, S.-R. Huang, C.-W. Huang, Y.-R. Cheng, C.-C. Chen, J. Hsiao, C.-W. Chen, and F. Lai, "A Progressively Expanded Database for Automated Lung Sound Analysis: An Update," Appl. Sci., vol. 12, no. 15, p. 7623, Jul. 2022, doi: [10.3390/app12157623](https://www.researchgate.net/publication/362327903_A_Progressively_Expanded_Database_for_Automated_Lung_Sound_Analysis_An_Update).
